# Customer Churn — Gradient Boosting vs Baselines (Realistic Generator + Optional XGBoost)

We build a production-style churn model on a realistic synthetic dataset with categorical + numeric features. We compare baselines (logistic regression) to boosted trees (HistGB / XGBoost if available), with CV and tuning.

**Author:** Olivier Robert-Duboille

**What you'll practice**
- reproducible data loading
- EDA with clear plots
- feature engineering / preprocessing pipelines
- cross-validation + hyperparameter tuning
- baseline vs advanced model comparison

This notebook is part of the *advanced-ml-mastery-collection* and is designed to be reproducible and portfolio-ready.

In [ ]:
# Reproducibility
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Plot settings
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_context('talk')


## Create a realistic churn dataset

Generate a churn-like dataset with meaningful nonlinear drivers and categorical effects.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(SEED)
n = 12000

tenure = rng.integers(0, 72, size=n)
monthly_charges = np.clip(rng.normal(70, 25, size=n), 15, 150)
contract = rng.choice(['month-to-month', 'one-year', 'two-year'], p=[0.58, 0.25, 0.17], size=n)
internet = rng.choice(['dsl', 'fiber', 'none'], p=[0.32, 0.55, 0.13], size=n)
support_calls = rng.poisson(1.1, size=n)
autopay = rng.choice(['yes', 'no'], p=[0.55, 0.45], size=n)
senior = rng.choice([0,1], p=[0.84, 0.16], size=n)

df = pd.DataFrame({
    'tenure_months': tenure,
    'monthly_charges': monthly_charges,
    'contract': contract,
    'internet': internet,
    'support_calls': support_calls,
    'autopay': autopay,
    'senior': senior,
})

# Churn probability model
logit = (
    -1.8
    + 0.03*(df['monthly_charges'] - 70)
    - 0.04*df['tenure_months']
    + 0.55*df['support_calls']
)
logit += df['contract'].map({'month-to-month': 0.9, 'one-year': -0.2, 'two-year': -0.7}).astype(float)
logit += df['internet'].map({'fiber': 0.25, 'dsl': 0.05, 'none': -0.5}).astype(float)
logit += df['autopay'].map({'yes': -0.25, 'no': 0.15}).astype(float)
logit += 0.25*df['senior']

p = 1/(1+np.exp(-logit))
df['churn'] = (rng.uniform(size=n) < p).astype(int)

display(df.head())
print('Churn rate:', df['churn'].mean())


## EDA

Explore distributions, relationships, and potential issues (missing values, skew, outliers).

In [ ]:
plt.figure(figsize=(6,3))
sns.countplot(x='churn', data=df)
plt.title('Churn distribution')
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x='churn', y='tenure_months', data=df)
plt.title('Tenure vs churn')
plt.show()


## Modeling: baselines vs boosted trees

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier

X = df.drop(columns=['churn'])
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

num_cols = ['tenure_months','monthly_charges','support_calls','senior']
cat_cols = ['contract','internet','autopay']

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])
ct = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

dummy = Pipeline([('prep', ct), ('model', DummyClassifier(strategy='most_frequent'))])
logreg = Pipeline([('prep', ct), ('model', LogisticRegression(max_iter=5000, solver='liblinear', random_state=SEED))])
hgb = Pipeline([('prep', ct), ('model', HistGradientBoostingClassifier(random_state=SEED))])

scoring = {'auc': 'roc_auc', 'f1': 'f1', 'acc': 'accuracy'}
rows = []
for name, model in [('dummy', dummy), ('logreg', logreg), ('hgb', hgb)]:
    res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    rows.append({
        'model': name,
        'auc_mean': res['test_auc'].mean(),
        'f1_mean': res['test_f1'].mean(),
        'acc_mean': res['test_acc'].mean(),
    })

import pandas as pd
display(pd.DataFrame(rows).sort_values('auc_mean', ascending=False))


## Hyperparameter tuning (HistGB) + optional XGBoost

In [ ]:
import numpy as np
from sklearn.model_selection import RandomizedSearchCV

space = {
    'model__learning_rate': [0.02, 0.05, 0.1],
    'model__max_depth': [None, 3, 5, 8],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__min_samples_leaf': [20, 50, 100],
    'model__l2_regularization': [0.0, 1e-3, 1e-2, 1e-1],
}
rs = RandomizedSearchCV(hgb, space, n_iter=25, scoring='roc_auc', cv=cv, n_jobs=-1, random_state=SEED)
rs.fit(X_train, y_train)
print('Best HGB AUC (CV):', rs.best_score_)
print('Best params:', rs.best_params_)

# Optional: XGBoost if installed
try:
    import xgboost as xgb
    xgb_clf = xgb.XGBClassifier(
        n_estimators=600,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=SEED,
        n_jobs=-1,
        eval_metric='auc',
    )
    xgb_pipe = Pipeline([('prep', ct), ('model', xgb_clf)])
    xgb_res = cross_validate(xgb_pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    print('XGBoost AUC (CV mean):', xgb_res['test_score'].mean())
except Exception as e:
    print('XGBoost not available in this environment:', repr(e))


## Final evaluation + ROC

In [ ]:
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

best = rs.best_estimator_
best.fit(X_train, y_train)
proba = best.predict_proba(X_test)[:,1]
pred = best.predict(X_test)

print('Test AUC:', roc_auc_score(y_test, proba))

RocCurveDisplay.from_predictions(y_test, proba)
plt.title('ROC (test)')
plt.show()

ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.title('Confusion matrix (test)')
plt.show()
